# 01 - r5 OpenAlex fetch (resume)

**Runtime: free CPU is fine - this job is network-bound, no GPU.** Run it in
parallel with the A100 notebook. Resumes from the bundled cache
(`MIGRATION_STATE.json` records 71,000/89,374 authors done locally; the
remainder plus anything in `retry_list.json` is fetched here). Politeness
identical to the local run: shared token bucket at 8 req/s, 6 workers,
transient-error retries, per-1000-author shards.

In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/who-inherits")  # for colab_common if bundle not yet unzipped
try:
    import colab_common as cc
except ImportError:
    # colab_common ships inside the bundle; bootstrap: mount, unzip, import
    from google.colab import drive as _d; _d.mount("/content/drive")
    import subprocess
    subprocess.run(["unzip", "-q", "-o",
                    "/content/drive/MyDrive/who-inherits/colab_bundle.zip",
                    "-d", "/content/work"], check=True)
    sys.path.insert(0, "/content/work/colab")
    import colab_common as cc
else:
    cc.mount_drive()
sys.path.insert(0, "/content/work/colab")
import colab_common as cc
cc.setup_workspace()
cc.verify_frozen_hashes()
print(cc.run_meta())

In [ ]:
cc.export_api_key()   # DRIVE/.openalex_key -> env; never printed
state = __import__("json").load(open("/content/work/results/robustness/MIGRATION_STATE.json"))
print({k: state["fetch_r5"][k] for k in ("authors_total", "authors_cached", "authors_remaining")})

In [ ]:
# retry_list first: authors whose cached works were empty after the local
# repair pass (empty list => nothing to do; the local pass already cleared 48)
import json
retry = json.load(open("/content/work/results/robustness/retry_list.json"))
print(f"{len(retry)} authors on the retry list")
if retry:
    cc.run_script("code/r5_fetch_author_works.py", args=("--repair",))
    cc.sync_to_drive()

In [ ]:
ckpt = cc.start_checkpoint_thread()   # Drive sync every 10 min
cc.run_script("code/r5_fetch_author_works.py")   # skips cached authors
cc.stop_checkpoint_thread()
cc.sync_to_drive()

In [ ]:
# repair pass on the completed cache, then the no-empty-authors assertion
cc.run_script("code/r5_fetch_author_works.py", args=("--repair",))
import pandas as pd, json
from pathlib import Path
cache = Path("/content/work/results/robustness/openalex_cache")
works = {}
for p in sorted(cache.glob("works_*.parquet")):
    df = pd.read_parquet(p)
    for a, w in zip(df.aid, df.works_json):
        works[a] = w
ids = set()
for f in ["chemistry", "econ", "math", "neuro", "physics"]:
    d = pd.read_parquet(f"/content/work/data/clean_dataset_{f}.parquet",
                        columns=["st_openalex_id", "adv_openalex_id"])
    ids |= set(d.st_openalex_id) | set(d.adv_openalex_id)
missing = sorted(a for a in ids if a not in works)
empty = sorted(a for a in ids if works.get(a) == "[]")
print(f"cached {len(ids) - len(missing)}/{len(ids)} | still empty: {len(empty)}")
assert not missing, f"{len(missing)} authors never fetched"
assert not empty, f"{len(empty)} frozen-table authors still cached empty"
print(f"[01] fetch complete: {len(ids)} authors, all with non-empty works")

In [ ]:
cc.write_done_flag("fetch")